## 10월 데이터용 버전
- 이 노트북은 2019-Oct.csv 기준입니다.
- 테이블(데이터프레임)명에 `_10` 접미사를 붙인 사본입니다.



In [1]:
!pip install geopandas

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import math
import platform
import geopandas as gpd

pd.set_option('display.float_format',"{:.2f}".format)

# OS에 따라 다른 폰트 지정
if platform.system() == 'Darwin':   # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
elif platform.system() == 'Windows':  # Windows
    plt.rcParams['font.family'] = 'Malgun Gothic'
else:  # Linux (예: Colab, Ubuntu)
    plt.rcParams['font.family'] = 'NanumGothic'

df0_10 = pd.read_csv("2019-Oct.csv")
print(df0_10.shape)
print(df0_10.head())
print(df0_10.columns.tolist())


(42448764, 9)
                event_time event_type  product_id          category_id  \
0  2019-10-01 00:00:00 UTC       view    44600062  2103807459595387724   
1  2019-10-01 00:00:00 UTC       view     3900821  2053013552326770905   
2  2019-10-01 00:00:01 UTC       view    17200506  2053013559792632471   
3  2019-10-01 00:00:01 UTC       view     1307067  2053013558920217191   
4  2019-10-01 00:00:04 UTC       view     1004237  2053013555631882655   

                         category_code     brand   price    user_id  \
0                                  NaN  shiseido   35.79  541312140   
1  appliances.environment.water_heater      aqua   33.20  554748717   
2           furniture.living_room.sofa       NaN  543.10  519107250   
3                   computers.notebook    lenovo  251.74  550050854   
4               electronics.smartphone     apple 1081.98  535871217   

                           user_session  
0  72d76fde-8bb3-4e00-8c23-a032dfed738c  
1  9333dfbd-b87a-4708-9857-633

In [2]:
df0_10.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42448764 entries, 0 to 42448763
Data columns (total 9 columns):
 #   Column         Dtype  
---  ------         -----  
 0   event_time     object 
 1   event_type     object 
 2   product_id     int64  
 3   category_id    int64  
 4   category_code  object 
 5   brand          object 
 6   price          float64
 7   user_id        int64  
 8   user_session   object 
dtypes: float64(1), int64(3), object(5)
memory usage: 2.8+ GB


In [3]:
smartphone_rows_10 = (df0_10['category_code'] == 'electronics.smartphone').sum()

ratio = smartphone_rows_10 / len(df0_10) * 100

print("전체 행 수:", len(df0_10))
print("스마트폰 행 수:", smartphone_rows_10)
print(f"전체 대비 스마트폰 비중: {ratio:.2f}%")

전체 행 수: 42448764
스마트폰 행 수: 11507231
전체 대비 스마트폰 비중: 27.11%


# 공통전처리1. 완전 중복 행 제거

In [4]:
# 5. 완전 중복 행 제거
df_10 = df0_10.drop_duplicates().copy()

In [5]:
print("중복 제거 전 행 수:", df0_10.shape[0])
print("중복 제거 후 행 수:", df_10.shape[0])
print("제거된 행 수:", df0_10.shape[0] - df_10.shape[0])
print("전체대비 삭제비율:", round((df0_10.shape[0] - df_10.shape[0]) / df0_10.shape[0] * 100, 2), "%")

중복 제거 전 행 수: 42448764
중복 제거 후 행 수: 42418544
제거된 행 수: 30220
전체대비 삭제비율: 0.07 %


# 공통전처리2. 동일고객이 동일세션에서 동일한 상품을 반복해서 구매할때

- 카테고리별 중복구매수 iqr 상한값을 구한 후
- 동일유저 + 동일상품 + 동일 세션에서 각 카테고리별 상한값 초과 반복 구매한 세션을 골라
- 짧은시간(0~60초) 안에 재 구매한 세션을 최종 이상후보 세션이라고 가정하였다

### 카테고리 공통 전처리후 실시해주세요 (현재는 전부 unknown 으로 처리)

In [6]:
# purchase 행만 추출
df_purchase_10 = df_10[df_10['event_type'] == 'purchase'].copy()

# category_code 결측치 처리
df_purchase_10['category_code'] = df_purchase_10['category_code'].fillna('unknown')

# 동일 유저 + 동일 상품 + 동일 세션 기준 구매 횟수 계산
purchase_sess_10 = (
    df_purchase_10
    .groupby(['user_id', 'product_id', 'user_session'])
    .agg(
        buy_cnt=('event_type', 'size'),
        category_code=('category_code', lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'unknown')
    )
    .reset_index()
)

# 2회 이상 반복 구매한 경우만 추출
purchase_sess_2_10 = purchase_sess_10[purchase_sess_10['buy_cnt'] >= 2].copy()

# 카테고리별 중복 구매수 IQR 상한값 계산
cat_iqr_10 = (
    purchase_sess_2_10
    .groupby('category_code')['buy_cnt']
    .agg(
        q1=lambda x: x.quantile(0.25),
        q3=lambda x: x.quantile(0.75)
    )
    .reset_index()
)

cat_iqr_10['iqr'] = cat_iqr_10['q3'] - cat_iqr_10['q1']
cat_iqr_10['upper'] = cat_iqr_10['q3'] + 1.5 * cat_iqr_10['iqr']

purchase_sess_2_10 = purchase_sess_2_10.merge(
    cat_iqr_10[['category_code', 'upper']],
    on='category_code',
    how='left'
)

purchase_sess_out_10 = purchase_sess_2_10[
    purchase_sess_2_10['buy_cnt'] > purchase_sess_2_10['upper']
].copy()

print("동일 유저 + 동일 상품 + 동일 세션 기준 2회 이상 반복 구매 조합 수:", len(purchase_sess_2_10))
print("카테고리별 IQR 상한값 초과 조합 수:", len(purchase_sess_out_10))
print("전체 2회 이상 조합 대비 비율:", round(len(purchase_sess_out_10) / len(purchase_sess_2_10) * 100, 2), "%")

display(purchase_sess_out_10.head())

동일 유저 + 동일 상품 + 동일 세션 기준 2회 이상 반복 구매 조합 수: 41340
카테고리별 IQR 상한값 초과 조합 수: 7142
전체 2회 이상 조합 대비 비율: 17.28 %


,user_id,product_id,user_session,buy_cnt,category_code,upper
12,493690536,1802037,7bac1c66-e281-4d45-86ee-9bd5a133116f,3,electronics.video.tv,2.00
14,502364367,5100816,e3609cf4-1ede-423a-b61b-cb2cfff6900e,3,unknown,2.00
16,506224239,1005192,d5bed622-0130-4c7c-99b6-2a1727dc5ffe,3,electronics.smartphone,2.00
19,512363791,1004739,73f6a0a1-afd7-4212-9728-425c3a48ab5b,3,electronics.smartphone,2.00
21,512363879,1003312,83db5b5a-2638-4296-b2d9-463dd589ecaa,4,electronics.smartphone,2.00


In [7]:
# 카테고리별 최소/최대 iqr값
 
cat_min_max_10 = (
    purchase_sess_out_10
    .groupby('category_code')['buy_cnt']
    .agg(['min', 'max'])
    .reset_index()
)

display(cat_min_max_10.head(20))
print("이상 조합의 최소 구매 횟수:", purchase_sess_out_10['buy_cnt'].min())
print("이상 조합의 최대 구매 횟수:", purchase_sess_out_10['buy_cnt'].max())

# 카테고리별 IQR 상한값에서
# 몇 회부터 이상치가 되는지 계산
cat_cut_10 = cat_iqr_10.copy()
cat_cut_10['out_start'] = np.floor(cat_cut_10['upper']).astype(int) + 1

# 횟수별로 카테고리가 몇 개인지 보기
cat_cut_cnt_10 = (
    cat_cut_10
    .groupby('out_start')['category_code']
    .nunique()
    .reset_index(name='cat_cnt')
    .sort_values('out_start')
)

display(cat_cut_cnt_10)


,category_code,min,max
0,accessories.bag,3,3
1,accessories.wallet,6,6
2,apparel.costume,3,3
3,apparel.shoes,3,4
4,apparel.shoes.keds,3,4
5,appliances.environment.air_conditioner,3,3
6,appliances.environment.air_heater,3,5
7,appliances.environment.vacuum,3,18
8,appliances.environment.water_heater,3,15
9,appliances.iron,3,4


이상 조합의 최소 구매 횟수: 3
이상 조합의 최대 구매 횟수: 46


,out_start,cat_cnt
0,3,98
1,4,3
2,5,4
3,6,2


한 세션안에서 같은 카테고리의 같은 상품을 반복하여 구매한 횟수에대해 iqr 상한값을 적용 했을 때
98개의 카테고리는 3회구매부터 이상치로 잡혔다.

In [8]:
# 카테고리별 이상치 시작 횟수 계산
cat_cut_10 = cat_iqr_10.copy()
cat_cut_10['out_start'] = np.floor(cat_cut_10['upper']).astype(int) + 1

# 4, 5, 6회부터 이상치로 잡히는 카테고리만 보기
cat_cut_456_10 = cat_cut_10[cat_cut_10['out_start'].isin([4, 5, 6])].copy()

cat_cut_456_10 = cat_cut_456_10.sort_values(['out_start', 'category_code'])

display(cat_cut_456_10[['category_code', 'upper', 'out_start']])


,category_code,upper,out_start
9,apparel.sock,3.00,4
20,appliances.kitchen.coffee_grinder,3.50,4
46,auto.accessories.winch,3.25,4
1,accessories.wallet,4.50,5
22,appliances.kitchen.dishwasher,4.50,5
74,electronics.audio.microphone,4.50,5
78,electronics.camera.video,4.50,5
14,appliances.environment.fan,5.12,6
57,computers.peripherals.camera,5.12,6


이상치가 4,5,6 개로 잡힌 항목은 위와같으며, 스마트폰의 이상후보 조합은 아래와 같았다

In [9]:
purchase_sess_out_10[purchase_sess_out_10['category_code'] == 'electronics.smartphone'] \
    .sort_values('buy_cnt', ascending=False)

,user_id,product_id,user_session,buy_cnt,category_code,upper
17424,530834332,1005074,ec97b68f-7cb9-480a-b2de-eca45a21c814,31,electronics.smartphone,2.00
25805,548256638,1005113,7df26835-e575-4a07-8c57-480922043935,24,electronics.smartphone,2.00
29422,553431815,1004247,c04e8f89-1fa6-4c17-b185-e00e318c2a6f,19,electronics.smartphone,2.00
40624,564749032,1004767,06680c58-8084-492c-ac8e-3ec47b03f1c0,18,electronics.smartphone,2.00
40085,564068124,1004767,34b9eb67-4755-49c3-bd42-5384ac9b6325,17,electronics.smartphone,2.00
...,...,...,...,...,...,...
41218,565952963,1005115,ef996d2e-9f93-4467-8fc1-9cd3cbaf2b4b,3,electronics.smartphone,2.00
41191,565830816,1003304,e26743d6-a787-40de-b933-a820972ff4a3,3,electronics.smartphone,2.00
41188,565825439,1005105,80df61d4-3459-4af3-bc75-e2f803775b11,3,electronics.smartphone,2.00
41184,565814035,1004870,7f5a1453-f199-450e-adef-7b1939b9bf8c,3,electronics.smartphone,2.00


#### 스마트폰 카테고리는 IQR 상한값이 2.0으로 계산되어, 3회 구매부터 1차 이상 후보로 분류되었지만 실제 최대 반복 구매 횟수는 31회로 나타나 상한값과의 격차가 매우 컸다.

In [10]:
# 카테고리별로 실제 최대 구매횟수와 IQR 상한값 차이 계산
cat_gap_10 = (
    purchase_sess_out_10
    .groupby('category_code')['buy_cnt']
    .max()
    .reset_index(name='max_buy_cnt')
    .merge(cat_iqr_10[['category_code', 'upper']], on='category_code', how='left')
)

cat_gap_10['over_gap'] = cat_gap_10['max_buy_cnt'] - cat_gap_10['upper']

# 상한값보다 과도하게 큰 카테고리 top 20
cat_gap_top20_base_10 = cat_gap_10.sort_values('over_gap', ascending=False).head(20).copy()

cat_gap_top20_base_10['all_cnt'] = cat_gap_top20_base_10['category_code'].map(
    purchase_sess_out_10.groupby('category_code').size()
).fillna(0).astype(int)

display(cat_gap_top20_base_10)


,category_code,max_buy_cnt,upper,over_gap,all_cnt
62,electronics.clocks,46,2.00,44.00,181
63,electronics.smartphone,31,2.00,29.00,4155
64,electronics.tablet,22,2.00,20.00,59
85,unknown,20,2.00,18.00,1170
7,appliances.environment.vacuum,18,2.00,16.00,64
49,construction.tools.drill,18,2.00,16.00,13
43,computers.notebook,16,2.00,14.00,171
31,auto.accessories.player,15,2.00,13.00,42
56,electronics.audio.headphone,15,2.00,13.00,357
8,appliances.environment.water_heater,15,2.00,13.00,18


#### iqr 상한값보다 과도하게 큰 카테고리 top 20위, 전자제품들이 다수 포함되어있었다


### 해당 상품들을 전부 이상치라고 볼수 없기때문에 두가지 조건을 줬다
### 1. 뷰, 카트 없이 "구매" 만 반복된 행 확인


In [11]:
# 수정

df_10_merge = df_10.copy()
df_10_merge['category_code'] = df_10_merge['category_code'].fillna('unknown')

out_log_10 = df_10_merge.merge(
    purchase_sess_out_10[['user_id', 'product_id', 'user_session']].drop_duplicates(),
    on=['user_id', 'product_id', 'user_session'],
    how='inner'
)

out_check_10 = (
    out_log_10
    .groupby(['category_code', 'user_id', 'product_id', 'user_session', 'event_type'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for col in ['view', 'cart', 'purchase']:
    if col not in out_check_10.columns:
        out_check_10[col] = 0

out_purchase_only_10 = out_check_10[
    (out_check_10['view'] == 0) &
    (out_check_10['cart'] == 0) &
    (out_check_10['purchase'] >= 2)
].copy()

print("purchase_sess_out_10 조합 수:", len(purchase_sess_out_10))
print("merge 후 확인된 조합 수:", out_check_10[['category_code','user_id','product_id','user_session']].drop_duplicates().shape[0])
print("view/cart 없이 purchase만 반복된 조합 수:", len(out_purchase_only_10))

display(out_purchase_only_10.head(20))

purchase_sess_out_10 조합 수: 7142
merge 후 확인된 조합 수: 7142
view/cart 없이 purchase만 반복된 조합 수: 0


event_type,category_code,user_id,product_id,user_session,cart,purchase,view


(유저id + 세션 + product_id) 조합 중 view, cart 없이 연속으로 구매한 세션은 없었다

### 2. 같은 유저 / 같은 세션 / 같은 상품 내에서 purchase 사이의 시간 간격을 구간별로 나눠서 확인하여 이상치 후보를 결정하였다

In [12]:
# purchase_sess_out_10 에 해당하는 purchase 행만 다시 가져오기
outlier_buy_10 = df_purchase_10.merge(
    purchase_sess_out_10[['user_id', 'product_id', 'user_session']],
    on=['user_id', 'product_id', 'user_session'],
    how='inner'
).copy()

# 시간형식 변환
outlier_buy_10['event_time'] = pd.to_datetime(outlier_buy_10['event_time'])

# 같은 유저 + 상품 + 세션 기준 시간순 정렬
outlier_buy_10 = outlier_buy_10.sort_values(
    ['user_id', 'user_session', 'product_id', 'event_time']
).copy()

# 바로 이전 purchase 시점 구하기
outlier_buy_10['prev_time'] = outlier_buy_10.groupby(
    ['user_id', 'user_session', 'product_id']
)['event_time'].shift(1)

# 재구매 간격(초) 계산
outlier_buy_10['gap_sec'] = (
    outlier_buy_10['event_time'] - outlier_buy_10['prev_time']
).dt.total_seconds()

# 실제 재구매 간격이 있는 purchase만 보기
gap_log_10 = outlier_buy_10[outlier_buy_10['gap_sec'].notna()].copy()

# 구간별 분포표 만들기
gap_log_10['gap_range'] = pd.cut(
    gap_log_10['gap_sec'],
    bins=[0, 30, 60, 180, 300, 600, 1800, 3600, 10800, 86400, float('inf')],
    right=False,
    include_lowest=True
)

gap_dist_10 = (
    gap_log_10['gap_range']
    .value_counts(sort=False)
    .rename_axis('gap_range_sec')
    .reset_index(name='count')
)

gap_dist_10['ratio_pct'] = (
    gap_dist_10['count'] / gap_dist_10['count'].sum() * 100
).round(2)

display(gap_dist_10)


,gap_range_sec,count,ratio_pct
0,"[0.0, 30.0)",260,1.43
1,"[30.0, 60.0)",4215,23.20
2,"[60.0, 180.0)",10207,56.18
3,"[180.0, 300.0)",1660,9.14
4,"[300.0, 600.0)",1058,5.82
5,"[600.0, 1800.0)",483,2.66
6,"[1800.0, 3600.0)",125,0.69
7,"[3600.0, 10800.0)",98,0.54
8,"[10800.0, 86400.0)",31,0.17
9,"[86400.0, inf)",33,0.18


#### 60~180 구간부터 대부분을 차지하기때문에, 재구매 이상치 후보를 0초 ~ 60초 사이라고 생각하였고 해당 구간을 10초간격으로 세분화 하여 표를 확인했다

In [13]:
gap_log_10 = df_purchase_10.merge(
    purchase_sess_out_10[['user_id', 'product_id', 'user_session']].drop_duplicates(),
    on=['user_id', 'product_id', 'user_session'],
    how='inner'
).copy()

gap_log_10['event_time'] = pd.to_datetime(gap_log_10['event_time'])

gap_log_10 = gap_log_10.sort_values(
    ['category_code', 'user_id', 'product_id', 'user_session', 'event_time']
).copy()

gap_log_10['prev_time'] = gap_log_10.groupby(
    ['user_id', 'product_id', 'user_session']
)['event_time'].shift(1)

gap_log_10['gap_sec'] = (
    gap_log_10['event_time'] - gap_log_10['prev_time']
).dt.total_seconds()

gap_log_10 = gap_log_10[
    gap_log_10['gap_sec'].notna() &
    (gap_log_10['gap_sec'] >= 0) &
    (gap_log_10['gap_sec'] < 60)
].copy()

gap_log_10['gap_range'] = pd.cut(
    gap_log_10['gap_sec'],
    bins=[0, 10, 20, 30, 40, 50, 60],
    right=False,
    include_lowest=True
)

gap_dist_10 = (
    gap_log_10['gap_range']
    .value_counts(sort=False)
    .rename_axis('gap_range_sec')
    .reset_index(name='count')
)

gap_dist_10['ratio_pct'] = (
    gap_dist_10['count'] / gap_dist_10['count'].sum() * 100
).round(2)

display(gap_dist_10)


,gap_range_sec,count,ratio_pct
0,"[0, 10)",22,0.49
1,"[10, 20)",83,1.85
2,"[20, 30)",155,3.46
3,"[30, 40)",585,13.07
4,"[40, 50)",1523,34.03
5,"[50, 60)",2107,47.08


#### 0~30초구간 상세

In [14]:
# 0~30초 구간: 기본 데이터 초기화
cat_gap_top20_10 = cat_gap_top20_base_10.copy()  # 원본 보존

cat_gap_top20_10['all_cnt'] = cat_gap_top20_10['category_code'].map(
    purchase_sess_out_10
    .groupby('category_code')
    .size()
).fillna(0).astype(int)

cat_gap_top20_10['drop_30_cnt'] = cat_gap_top20_10['category_code'].map(
    purchase_sess_out_10
    .merge(
        gap_log_10.loc[
            (gap_log_10['gap_sec'] >= 0) & (gap_log_10['gap_sec'] < 30),
            ['user_id', 'product_id', 'user_session']
        ].drop_duplicates(),
        on=['user_id', 'product_id', 'user_session'],
        how='inner'
    )
    .groupby('category_code')
    .size()
).fillna(0).astype(int)

cat_gap_top20_10['keep_30_cnt'] = cat_gap_top20_10['all_cnt'] - cat_gap_top20_10['drop_30_cnt']

cat_gap_top20_10['max_after_30'] = cat_gap_top20_10['category_code'].map(
    purchase_sess_out_10
    .merge(
        gap_log_10.loc[
            (gap_log_10['gap_sec'] >= 0) & (gap_log_10['gap_sec'] < 30),
            ['user_id', 'product_id', 'user_session']
        ].drop_duplicates(),
        on=['user_id', 'product_id', 'user_session'],
        how='left',
        indicator=True
    )
    .query("_merge == 'left_only'")
    .groupby('category_code')['buy_cnt']
    .max()
).fillna(0).astype(int)

display(
    cat_gap_top20_10[
        ['category_code', 'upper', 'max_buy_cnt', 'max_after_30', 'all_cnt', 'drop_30_cnt', 'keep_30_cnt']
    ]
)



,category_code,upper,max_buy_cnt,max_after_30,all_cnt,drop_30_cnt,keep_30_cnt
62,electronics.clocks,2.00,46,46,181,6,175
63,electronics.smartphone,2.00,31,31,4155,104,4051
64,electronics.tablet,2.00,22,22,59,3,56
85,unknown,2.00,20,14,1170,53,1117
7,appliances.environment.vacuum,2.00,18,18,64,2,62
49,construction.tools.drill,2.00,18,18,13,0,13
43,computers.notebook,2.00,16,16,171,6,165
31,auto.accessories.player,2.00,15,15,42,2,40
56,electronics.audio.headphone,2.00,15,15,357,17,340
8,appliances.environment.water_heater,2.00,15,15,18,0,18


### 표 해석방법
### electronics.clocks 기준 0~30초 조합을 삭제해도 가장 많이구매한 46회가 유지되었고
### 전체 조합중 제거되는조합은 6개뿐이였다
### 각 구간별로 확인해봤다 (~60초까지)

#### 0~40초구간

In [15]:
# 0~40초 구간: 기본 데이터 초기화
cat_gap_top20_10 = cat_gap_top20_base_10.copy()  # 원본 보존

cat_gap_top20_10['drop_40_cnt'] = cat_gap_top20_10['category_code'].map(
    purchase_sess_out_10
    .merge(
        gap_log_10.loc[
            (gap_log_10['gap_sec'] >= 0) & (gap_log_10['gap_sec'] < 40),
            ['user_id', 'product_id', 'user_session']
        ].drop_duplicates(),
        on=['user_id', 'product_id', 'user_session'],
        how='inner'
    )
    .groupby('category_code')
    .size()
).fillna(0).astype(int)

cat_gap_top20_10['keep_40_cnt'] = cat_gap_top20_10['all_cnt'] - cat_gap_top20_10['drop_40_cnt']

cat_gap_top20_10['max_after_40'] = cat_gap_top20_10['category_code'].map(
    purchase_sess_out_10
    .merge(
        gap_log_10.loc[
            (gap_log_10['gap_sec'] >= 0) & (gap_log_10['gap_sec'] < 40),
            ['user_id', 'product_id', 'user_session']
        ].drop_duplicates(),
        on=['user_id', 'product_id', 'user_session'],
        how='left',
        indicator=True
    )
    .query("_merge == 'left_only'")
    .groupby('category_code')['buy_cnt']
    .max()
).fillna(0).astype(int)

display(
    cat_gap_top20_10[
        ['category_code', 'upper', 'max_buy_cnt', 'max_after_40', 'all_cnt', 'drop_40_cnt', 'keep_40_cnt']
    ]
)


,category_code,upper,max_buy_cnt,max_after_40,all_cnt,drop_40_cnt,keep_40_cnt
62,electronics.clocks,2.00,46,46,181,19,162
63,electronics.smartphone,2.00,31,24,4155,334,3821
64,electronics.tablet,2.00,22,6,59,13,46
85,unknown,2.00,20,14,1170,147,1023
7,appliances.environment.vacuum,2.00,18,18,64,5,59
49,construction.tools.drill,2.00,18,18,13,2,11
43,computers.notebook,2.00,16,16,171,26,145
31,auto.accessories.player,2.00,15,15,42,6,36
56,electronics.audio.headphone,2.00,15,15,357,51,306
8,appliances.environment.water_heater,2.00,15,15,18,0,18


#### 0~50초구간

In [16]:
# 0~50초 구간: 기본 데이터 초기화
cat_gap_top20_10 = cat_gap_top20_base_10.copy()  # 원본 보존

cat_gap_top20_10['drop_50_cnt'] = cat_gap_top20_10['category_code'].map(
    purchase_sess_out_10
    .merge(
        gap_log_10.loc[
            (gap_log_10['gap_sec'] >= 0) & (gap_log_10['gap_sec'] < 50),
            ['user_id', 'product_id', 'user_session']
        ].drop_duplicates(),
        on=['user_id', 'product_id', 'user_session'],
        how='inner'
    )
    .groupby('category_code')
    .size()
).fillna(0).astype(int)

cat_gap_top20_10['keep_50_cnt'] = cat_gap_top20_10['all_cnt'] - cat_gap_top20_10['drop_50_cnt']

cat_gap_top20_10['max_after_50'] = cat_gap_top20_10['category_code'].map(
    purchase_sess_out_10
    .merge(
        gap_log_10.loc[
            (gap_log_10['gap_sec'] >= 0) & (gap_log_10['gap_sec'] < 50),
            ['user_id', 'product_id', 'user_session']
        ].drop_duplicates(),
        on=['user_id', 'product_id', 'user_session'],
        how='left',
        indicator=True
    )
    .query("_merge == 'left_only'")
    .groupby('category_code')['buy_cnt']
    .max()
).fillna(0).astype(int)

display(
    cat_gap_top20_10[
        ['category_code', 'upper', 'max_buy_cnt', 'max_after_50', 'all_cnt', 'drop_50_cnt', 'keep_50_cnt']
    ]
)


,category_code,upper,max_buy_cnt,max_after_50,all_cnt,drop_50_cnt,keep_50_cnt
62,electronics.clocks,2.00,46,46,181,54,127
63,electronics.smartphone,2.00,31,24,4155,913,3242
64,electronics.tablet,2.00,22,6,59,20,39
85,unknown,2.00,20,14,1170,286,884
7,appliances.environment.vacuum,2.00,18,6,64,13,51
49,construction.tools.drill,2.00,18,4,13,4,9
43,computers.notebook,2.00,16,16,171,61,110
31,auto.accessories.player,2.00,15,5,42,12,30
56,electronics.audio.headphone,2.00,15,9,357,109,248
8,appliances.environment.water_heater,2.00,15,5,18,5,13


#### 0~60초구간

In [17]:
# 0~60초 구간: 기본 데이터 초기화
cat_gap_top20_10 = cat_gap_top20_base_10.copy()  # 원본 보존

cat_gap_top20_10['drop_60_cnt'] = cat_gap_top20_10['category_code'].map(
    purchase_sess_out_10
    .merge(
        gap_log_10.loc[
            (gap_log_10['gap_sec'] >= 0) & (gap_log_10['gap_sec'] < 60),
            ['user_id', 'product_id', 'user_session']
        ].drop_duplicates(),
        on=['user_id', 'product_id', 'user_session'],
        how='inner'
    )
    .groupby('category_code')
    .size()
).fillna(0).astype(int)

cat_gap_top20_10['keep_60_cnt'] = cat_gap_top20_10['all_cnt'] - cat_gap_top20_10['drop_60_cnt']

cat_gap_top20_10['max_after_60'] = cat_gap_top20_10['category_code'].map(
    purchase_sess_out_10
    .merge(
        gap_log_10.loc[
            (gap_log_10['gap_sec'] >= 0) & (gap_log_10['gap_sec'] < 60),
            ['user_id', 'product_id', 'user_session']
        ].drop_duplicates(),
        on=['user_id', 'product_id', 'user_session'],
        how='left',
        indicator=True
    )
    .query("_merge == 'left_only'")
    .groupby('category_code')['buy_cnt']
    .max()
).fillna(0).astype(int)

display(
    cat_gap_top20_10[
        ['category_code', 'upper', 'max_buy_cnt', 'max_after_60', 'all_cnt', 'drop_60_cnt', 'keep_60_cnt']
    ]
)


,category_code,upper,max_buy_cnt,max_after_60,all_cnt,drop_60_cnt,keep_60_cnt
62,electronics.clocks,2.00,46,8,181,88,93
63,electronics.smartphone,2.00,31,17,4155,1578,2577
64,electronics.tablet,2.00,22,6,59,30,29
85,unknown,2.00,20,9,1170,459,711
7,appliances.environment.vacuum,2.00,18,4,64,29,35
49,construction.tools.drill,2.00,18,4,13,5,8
43,computers.notebook,2.00,16,6,171,89,82
31,auto.accessories.player,2.00,15,5,42,17,25
56,electronics.audio.headphone,2.00,15,9,357,176,181
8,appliances.environment.water_heater,2.00,15,5,18,7,11


In [18]:
cut_result_10 = pd.DataFrame({
    'cut_sec': [30, 40, 50, 60],
    'drop_pair_cnt': [
        len(gap_log_10.loc[(gap_log_10['gap_sec'] >= 0) & (gap_log_10['gap_sec'] < 30), ['user_id', 'product_id', 'user_session']].drop_duplicates()),
        len(gap_log_10.loc[(gap_log_10['gap_sec'] >= 0) & (gap_log_10['gap_sec'] < 40), ['user_id', 'product_id', 'user_session']].drop_duplicates()),
        len(gap_log_10.loc[(gap_log_10['gap_sec'] >= 0) & (gap_log_10['gap_sec'] < 50), ['user_id', 'product_id', 'user_session']].drop_duplicates()),
        len(gap_log_10.loc[(gap_log_10['gap_sec'] >= 0) & (gap_log_10['gap_sec'] < 60), ['user_id', 'product_id', 'user_session']].drop_duplicates()),
],
    'drop_buy_rows': [
        len(df_purchase_10.merge(gap_log_10.loc[(gap_log_10['gap_sec'] >= 0) & (gap_log_10['gap_sec'] < 30), ['user_id', 'product_id', 'user_session']].drop_duplicates(), on=['user_id', 'product_id', 'user_session'], how='inner')),
        len(df_purchase_10.merge(gap_log_10.loc[(gap_log_10['gap_sec'] >= 0) & (gap_log_10['gap_sec'] < 40), ['user_id', 'product_id', 'user_session']].drop_duplicates(), on=['user_id', 'product_id', 'user_session'], how='inner')),
        len(df_purchase_10.merge(gap_log_10.loc[(gap_log_10['gap_sec'] >= 0) & (gap_log_10['gap_sec'] < 50), ['user_id', 'product_id', 'user_session']].drop_duplicates(), on=['user_id', 'product_id', 'user_session'], how='inner')),
        len(df_purchase_10.merge(gap_log_10.loc[(gap_log_10['gap_sec'] >= 0) & (gap_log_10['gap_sec'] < 60), ['user_id', 'product_id', 'user_session']].drop_duplicates(), on=['user_id', 'product_id', 'user_session'], how='inner')),]
})

cut_result_10['drop_pair_ratio'] = (cut_result_10['drop_pair_cnt'] / len(purchase_sess_out_10) * 100).round(2)
cut_result_10['drop_buy_ratio'] = (cut_result_10['drop_buy_rows'] / len(df_purchase_10) * 100).round(4)

display(cut_result_10)


,cut_sec,drop_pair_cnt,drop_buy_rows,drop_pair_ratio,drop_buy_ratio
0,30,228,828,3.19,0.11
1,40,697,2677,9.76,0.36
2,50,1688,6507,23.63,0.88
3,60,2823,10748,39.53,1.45


#### 재구매 간격분포를 60초까지 확인해본결과 30초 이후 구간부터 cnt 갯수가 증가하였지만 상단 표를 확인해보면 반복 구매횟수도 60초구간부터 확연히 줄기 시작하므로 0~60초를 경계구간으로 해석하였고 이에 따라 플래그 컬럼을 만들어 후 분석목적에 맞게 적용하려 한다

In [19]:
pair_cols = ['user_id', 'product_id', 'user_session']

# ── Step 1: purchase만 반복된 세션 확인 (10월은 0건) ─────────
problem_sessions = out_purchase_only_10['user_session'].drop_duplicates()

if len(problem_sessions) > 0:
    session_view_check = (
        df_10[df_10['user_session'].isin(problem_sessions)]
        .groupby('user_session')['event_type']
        .apply(lambda x: (x == 'view').any())
        .reset_index(name='has_view')
    )
    del_sessions  = session_view_check[~session_view_check['has_view']]['user_session']
    keep_sessions = session_view_check[ session_view_check['has_view']]['user_session']
    df_10 = df_10[~df_10['user_session'].isin(del_sessions)].copy()
    gap_log_10 = gap_log_10[~gap_log_10['user_session'].isin(del_sessions)].copy()
else:
    keep_sessions = pd.Series([], dtype=str)

# ── Step 2: 플래그 초기화 ────────────────────────────────────
df_10 = df_10.drop(columns=[c for c in ['repeat_time_flag','f30','f60','fsec_10'] if c in df_10.columns], errors='ignore')
df_10['event_time']       = pd.to_datetime(df_10['event_time'])
gap_log_10['prev_time']   = pd.to_datetime(gap_log_10['prev_time'])
gap_log_10['event_time']  = pd.to_datetime(gap_log_10['event_time'])

# ── Step 3: gap_log 기반 [prev_time, cur_time] 구간 플래그 ──
df_10 = df_10.reset_index(drop=True)
df_10['_idx'] = df_10.index

t_0_30_10 = gap_log_10.loc[
    (gap_log_10['gap_sec'] >= 0) & (gap_log_10['gap_sec'] < 30),
    pair_cols + ['prev_time', 'event_time']
].rename(columns={'event_time': 'cur_time'})

t_30_60_10 = gap_log_10.loc[
    (gap_log_10['gap_sec'] >= 30) & (gap_log_10['gap_sec'] < 60),
    pair_cols + ['prev_time', 'event_time']
].rename(columns={'event_time': 'cur_time'})

idx_0_30 = (
    df_10.merge(t_0_30_10, on=pair_cols, how='inner')
    .query('event_time > prev_time and event_time <= cur_time')
    ['_idx'].unique()
)
idx_30_60 = (
    df_10.merge(t_30_60_10, on=pair_cols, how='inner')
    .query('event_time >= prev_time and event_time <= cur_time')
    ['_idx'].unique()
)

df_10['fsec_10'] = None
df_10.loc[idx_0_30,  'fsec_10'] = '0_30'
df_10.loc[idx_30_60, 'fsec_10'] = '30_60'

if len(keep_sessions) > 0:
    df_10.loc[df_10['user_session'].isin(keep_sessions), 'fsec_10'] = 'keep'

df_10 = df_10.drop(columns=['_idx'])

print(df_10['fsec_10'].value_counts(dropna=False))

# ── Step 4: 구간별 요약 테이블 ───────────────────────────────
total_view     = (df_10['event_type'] == 'view').sum()
total_cart     = (df_10['event_type'] == 'cart').sum()
total_purchase = (df_10['event_type'] == 'purchase').sum()

def make_summary(label, lo, hi):
    pairs = gap_log_10.loc[
        (gap_log_10['gap_sec'] >= lo) & (gap_log_10['gap_sec'] < hi),
        pair_cols
    ].drop_duplicates()

    log = df_10.merge(pairs, on=pair_cols, how='inner')
    v = (log['event_type'] == 'view').sum()
    c = (log['event_type'] == 'cart').sum()
    p = (log['event_type'] == 'purchase').sum()

    return {
        '구간': label,
        '조합수': len(pairs),
        '전체행수': len(log),
        'view수': v,
        'cart수': c,
        'purchase수': p,
        '전체 view 중 비율(%)':     round(v / total_view     * 100, 6) if total_view     > 0 else 0,
        '전체 cart 중 비율(%)':     round(c / total_cart     * 100, 6) if total_cart     > 0 else 0,
        '전체 purchase 중 비율(%)': round(p / total_purchase * 100, 6) if total_purchase > 0 else 0,
    }

summary_df_10 = pd.DataFrame([
    make_summary('0~30초',  0, 30),
    make_summary('30~60초', 30, 60),
])

display(summary_df_10)

fsec_10
None     42404949
30_60       13134
0_30          461
Name: count, dtype: int64


,구간,조합수,전체행수,view수,cart수,purchase수,전체 view 중 비율(%),전체 cart 중 비율(%),전체 purchase 중 비율(%)
0,0~30초,228,2491,1317,346,828,0.00,0.04,0.11
1,30~60초,2694,32702,16871,5502,10329,0.04,0.61,1.39
